## Bước 1: Load dữ liệu RFM đã tính, loại bỏ khách Monetary âm

In [1]:
import pandas as pd

rfm = pd.read_csv('rfm_customers.csv')
print(f"So khach hang ban dau: {len(rfm)}")

rfm = rfm[rfm['Monetary'] >= 0]
print(f"Sau khi loai khach Monetary am: {len(rfm)}")

So khach hang ban dau: 5881
Sau khi loai khach Monetary am: 5858


## Bước 2: Chấm điểm Recency (1-5) — CHÚ Ý: số ngày càng NHỎ thì điểm càng CAO (ngược lại Frequency/Monetary)

Monetary = tổng số tiền đã chi (đã trừ hàng hủy/trả) → số lớn = tốt

Recency = đã bao nhiêu ngày KHÔNG mua (tính từ lần mua cuối tới "mốc hôm nay" giả định) → số nhỏ = tốt (mới mua gần đây)

Frequency = đã mua bao nhiêu LẦN (bao nhiêu đơn hàng khác nhau) → số lớn = tốt



In [2]:
# qcut chia thanh 5 nhom bang nhau theo GIA TRI tang dan, gan nhan 1->5
# Vi Recency thap = tot, can DAO NGUOC thu tu nhan (labels=[5,4,3,2,1])
rfm['R_score'] = pd.qcut(rfm['Recency'], q=5, labels=[5, 4, 3, 2, 1]).astype(int)

print(rfm[['Recency', 'R_score']].sort_values('Recency').head(10))

      Recency  R_score
5092      1.0        5
5091      1.0        5
3420      1.0        5
2055      1.0        5
2363      1.0        5
2202      1.0        5
3024      1.0        5
677       1.0        5
5131      1.0        5
727       1.0        5


## Bước 3: Chấm điểm Frequency và Monetary (1-5) — giá trị càng LỚN điểm càng CAO

Monetary = tổng số tiền đã chi (đã trừ hàng hủy/trả) → số lớn = tốt

Recency = đã bao nhiêu ngày KHÔNG mua (tính từ lần mua cuối tới "mốc hôm nay" giả định) → số nhỏ = tốt (mới mua gần đây)

Frequency = đã mua bao nhiêu LẦN (bao nhiêu đơn hàng khác nhau) → số lớn = tốt



In [3]:
# Frequency thuong co nhieu gia tri trung nhau (VD: rat nhieu khach cung mua 1 lan)
# nen dung method='first' de tranh loi qcut khong chia duoc thanh 5 nhom bang nhau
rfm['F_score'] = pd.qcut(rfm['Frequency'].rank(method='first'), q=5, labels=[1, 2, 3, 4, 5]).astype(int)
rfm['M_score'] = pd.qcut(rfm['Monetary'].rank(method='first'), q=5, labels=[1, 2, 3, 4, 5]).astype(int)

print(rfm[['Frequency', 'F_score', 'Monetary', 'M_score']].head(10))

    Frequency  F_score  Monetary  M_score
1         8.0        4   4921.53        5
2         5.0        4   2019.40        4
3         4.0        3   4404.54        5
4         1.0        1    334.40        2
5         1.0        1    300.93        2
6        10.0        5   1889.21        4
7         2.0        2    406.76        2
8         1.0        1   1079.40        3
9         2.0        2    947.61        3
10        6.0        4   6371.73        5


## Bước 4: Kết hợp 3 điểm số thành nhãn phân khúc dễ hiểu

In [4]:
def gan_nhan_phan_khuc(row):
    r, f, m = row['R_score'], row['F_score'], row['M_score']

    if r >= 4 and f >= 4 and m >= 4:
        return 'VIP / Khach hang trung thanh'
    elif r <= 2 and f >= 4 and m >= 4:
        return 'Co nguy co roi bo (tung tot, lau khong quay lai)'
    elif r >= 4 and f <= 2:
        return 'Khach hang moi'
    elif r <= 2 and f <= 2 and m <= 2:
        return 'Da roi bo / Khong con hoat dong'
    else:
        return 'Khach hang trung binh'

rfm['Segment'] = rfm.apply(gan_nhan_phan_khuc, axis=1)

print(rfm['Segment'].value_counts())

Segment
Khach hang trung binh                               2616
VIP / Khach hang trung thanh                        1297
Da roi bo / Khong con hoat dong                     1275
Khach hang moi                                       445
Co nguy co roi bo (tung tot, lau khong quay lai)     225
Name: count, dtype: int64


## Bước 5: Lưu kết quả cuối cùng

In [5]:
rfm.to_csv('rfm_segments.csv', index=False, encoding='utf-8-sig')
print("Da luu vao rfm_segments.csv")
print(rfm.head(10))

Da luu vao rfm_segments.csv
    Customer ID  Monetary  Recency  Frequency  R_score  F_score  M_score  \
1       12347.0   4921.53      2.0        8.0        5        4        5   
2       12348.0   2019.40     75.0        5.0        3        4        4   
3       12349.0   4404.54     19.0        4.0        5        3        5   
4       12350.0    334.40    310.0        1.0        2        1        2   
5       12351.0    300.93    375.0        1.0        2        1        2   
6       12352.0   1889.21     36.0       10.0        4        5        4   
7       12353.0    406.76    204.0        2.0        2        2        2   
8       12354.0   1079.40    232.0        1.0        2        1        3   
9       12355.0    947.61    214.0        2.0        2        2        3   
10      12356.0   6371.73     23.0        6.0        4        4        5   

                            Segment  
1      VIP / Khach hang trung thanh  
2             Khach hang trung binh  
3             Khach h